In [10]:
import pandas as pd
import glob
import os
import gc

# 1. Define your exact folder paths
raw_folder = r"D:\Source\repo\flemms_digital_divide_project\data\raw"
processed_folder = r"D:\Source\repo\flemms_digital_divide_project\data\processed"

# Create partitioned directories to avoid memory crashes
hh_processed_folder = os.path.join(processed_folder, "households")
ind_processed_folder = os.path.join(processed_folder, "individuals")
os.makedirs(hh_processed_folder, exist_ok=True)
os.makedirs(ind_processed_folder, exist_ok=True)

# 2. Use a recursive wildcard (**) to find CSV files inside all subfolders
file_pattern = os.path.join(raw_folder, "**", "*.csv")
data_files = glob.glob(file_pattern, recursive=True)

# 3. Loop through each discovered file, read, and save IMMEDIATELY (no memory lists)
for file in data_files:
    parent_folder = os.path.basename(os.path.dirname(file))
    filename = os.path.basename(file).upper()
    
    # Skip the duplicate folder explicitly
    if "(1)" in parent_folder:
        continue
        
    print(f"Processing: {parent_folder} -> {filename}...")
    
    try:
        # FIX 1: encoding='latin-1' handles special characters like 'Ñ'
        # FIX 2: Removed low_memory=False to save RAM during the read phase
        df = pd.read_csv(file, on_bad_lines='skip', encoding='latin-1')
        
        # Add tracking columns for data lineage
        df['source_folder'] = parent_folder
        df['source_file'] = filename
        
        # Define a safe output filename based on the parent folder to avoid overwriting
        safe_filename = f"{parent_folder}_{filename}"
        
        # Route and SAVE immediately to free up memory
        if any(keyword in filename for keyword in ["HOUSEHOLD", "RT01", "RTF1", "HOUSING"]):
            out_path = os.path.join(hh_processed_folder, safe_filename)
            df.to_csv(out_path, index=False)
            print("  -> Saved to Households partition.")
            
        elif any(keyword in filename for keyword in ["POPULATION", "MEMBER", "RT02", "RTF2", "DEMOGRAPHIC", "INDIVIDUAL"]):
            out_path = os.path.join(ind_processed_folder, safe_filename)
            df.to_csv(out_path, index=False)
            print("  -> Saved to Individuals partition.")
            
        elif "2003" in filename:
            out_path = os.path.join(ind_processed_folder, safe_filename)
            df.to_csv(out_path, index=False)
            print("  -> Saved to Individuals partition (2003 format).")
            
        else:
            print(f"  -> Warning: Could not categorize {filename}. Skipping.")
            
        # FIX 3: Explicitly delete the dataframe from RAM and trigger garbage collection
        del df
        gc.collect()
        
    except Exception as e:
        print(f"Error loading {file}: {e}")

print("\nData Engineering Step Complete: Data successfully partitioned without crashing memory!")

Processing: PHL-NSO-FLEMMS-1994-PUF -> FLEMMS PUF 1994 HOUSEHOLD.CSV...
  -> Saved to Households partition.
Processing: PHL-NSO-FLEMMS-1994-PUF -> FLEMMS PUF 1994 POPULATION.CSV...
  -> Saved to Individuals partition.
Processing: PHL-PSA-FLEMMS-2003-PUF -> FLEMMS PUF 2003.CSV...
  -> Saved to Individuals partition (2003 format).
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS OSY PUF 2008 RT01 - ALL MEMBERS.CSV...
  -> Saved to Households partition.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS OSY PUF 2008 RT02 - 10-64 YEAR OLD.CSV...
  -> Saved to Individuals partition.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS PUF 2008 RT01 - ALL MEMBERS.CSV...
  -> Saved to Households partition.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS PUF 2008 RT02 - 10-64 YEARS OLD.CSV...
  -> Saved to Individuals partition.
Processing: Standard -> FLEMMS PUF 2013 STANDARD RT01 - HOUSEHOLD MEMBERS.CSV...
  -> Saved to Households partition.
Processing: Standard -> FLEMMS PUF 2013 STANDARD RT02 - INDIVIDUAL.

In [1]:
import pandas as pd
import os

# 1. INSPECT THE DIRECTORY CONTENTS
processed_folder = r"D:\Source\repo\flemms_digital_divide_project\data\processed"

print(f"=== FILES CURRENTLY IN: {processed_folder} ===")
# Use os.walk to see files inside subfolders (like /households and /individuals)
for root, dirs, files in os.walk(processed_folder):
    for file in files:
        # Print the full path of every file found
        print(os.path.join(root, file))
print("\n" + "="*60 + "\n")

# 2. LOAD A SPECIFIC FILE FOR EDA
# You can change this path to one of the household or individual files printed above
file_path = r"D:\Source\repo\flemms_digital_divide_project\data\processed\merged_flemms_metadata.csv"

# Safely check if the file exists before running the EDA
if os.path.exists(file_path):
    df = pd.read_csv(file_path)

    # 3. CORE VARIABLES INSPECTION
    print("=== 1. DATASET OVERVIEW & CORE VARIABLES ===")
    print(f"Total Rows: {df.shape[0]}")
    print(f"Total Columns: {df.shape[1]}\n")
    print("Column Names:")
    print(df.columns.tolist())

    print("\n--- First 5 Rows ---")
    display(df.head())

    # 4. MISSING VALUES CHECK
    print("\n=== 2. MISSING VALUES ANALYSIS ===")
    missing_df = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
    })
    display(missing_df.sort_values(by='Missing Count', ascending=False))

    # 5. DUPLICATE DATA CHECK
    print("\n=== 3. DUPLICATE ROWS CHECK ===")
    duplicate_count = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicate_count}")
    print(f"Percentage of duplicate rows: {(duplicate_count / len(df)) * 100:.2f}%")
else:
    print(f"WARNING: The file was not found at {file_path}")
    print("Please copy one of the file paths printed at the top and paste it into the 'file_path' variable.")

=== FILES CURRENTLY IN: D:\Source\repo\flemms_digital_divide_project\data\processed ===
D:\Source\repo\flemms_digital_divide_project\data\processed\merged_flemms_metadata.csv
D:\Source\repo\flemms_digital_divide_project\data\processed\merged_flemms_puf_data.csv
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-NSO-FLEMMS-1994-PUF_FLEMMS PUF 1994 HOUSEHOLD.CSV
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-PSA-FLEMMS-2008-PUF_FLEMMS OSY PUF 2008 RT01 - ALL MEMBERS.CSV
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-PSA-FLEMMS-2008-PUF_FLEMMS PUF 2008 RT01 - ALL MEMBERS.CSV
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-PSA-FLEMMS-2019-PUF_FLEMMS PUF 2019 HOUSEHOLD NAME.CSV
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-PSA-FLEMMS-2019-PUF_FLEMMS PUF 2019 HOUSEHOLD RECORD.CSV
D:\Source\repo\flemms_digital_divide_project\data\processed\households\PHL-PSA-

,QUEST,Questionnaire,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,source_year,FORM1,FLEMMS RECORD,FLEMMS08,Functional Literacy Education & Mass Media Survey November 2008,FLEMMS Form 1,FLEMMS_QUEST,FLEMMS Household Questionnaire
0,NaN,NaN,_IDS0,(Id Items),NaN,NaN,flemms_1994_metadata(dictionary).xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,FN,FN,flemms_1994_metadata(dictionary).xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,REGN,REGN,flemms_1994_metadata(dictionary).xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,PROV,PROV,flemms_1994_metadata(dictionary).xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,MUN,MUN,flemms_1994_metadata(dictionary).xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== 2. MISSING VALUES ANALYSIS ===


,Missing Count,Missing Percentage (%)
QUEST,684,100.000000
Questionnaire,684,100.000000
FORM1,684,100.000000
FLEMMS RECORD,684,100.000000
FLEMMS08,684,100.000000
Functional Literacy Education & Mass Media Survey November 2008,684,100.000000
FLEMMS Form 1,684,100.000000
FLEMMS_QUEST,684,100.000000
FLEMMS Household Questionnaire,684,100.000000
Unnamed: 2,660,96.491228



=== 3. DUPLICATE ROWS CHECK ===
Number of duplicate rows: 0
Percentage of duplicate rows: 0.00%


In [2]:
import pandas as pd
import os

# 1. Define path to the partitioned individuals folder
ind_processed_folder = r"D:\Source\repo\flemms_digital_divide_project\data\processed\individuals"

# 2. Grab the first available file in the individuals directory to prototype our cleaning
available_files = [f for f in os.listdir(ind_processed_folder) if f.endswith('.csv')]

if not available_files:
    print("No CSV files found in the individuals folder.")
else:
    sample_file = available_files[0]
    file_path = os.path.join(ind_processed_folder, sample_file)

    print(f"--- Prototyping ETL on: {sample_file} ---\n")
    
    # Load the data
    df = pd.read_csv(file_path, low_memory=False)
    initial_rows = len(df)
    initial_cols = len(df.columns)

    # ==========================================
    # DATA ENGINEERING: CLEANING & HARMONIZATION
    # ==========================================

    # A. Schema Standardization: Clean column names
    # Convert all headers to lowercase, replace spaces with underscores, and strip whitespace
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

    # B. Data Deduplication: Remove exact duplicate rows
    df = df.drop_duplicates()
    dropped_duplicates = initial_rows - len(df)

    # C. Handle Missing Values: Drop columns that are 100% empty
    df = df.dropna(axis=1, how='all')
    dropped_cols = initial_cols - len(df.columns)

    # ==========================================
    # REVIEW RESULTS
    # ==========================================
    print("=== POST-CLEANING OVERVIEW ===")
    print(f"Duplicate Rows Removed: {dropped_duplicates}")
    print(f"Completely Empty Columns Dropped: {dropped_cols}")
    print(f"Current Rows: {df.shape[0]}")
    print(f"Current Columns: {df.shape[1]}\n")
    
    print("Cleaned Column Names (Notice they are now lower_snake_case):")
    print(df.columns.tolist())

    print("\n--- First 5 Rows of Cleaned Data ---")
    display(df.head())

No CSV files found in the individuals folder.


In [3]:
import pandas as pd
import glob
import os
import gc

# 1. Define paths
raw_folder = r"D:\Source\repo\flemms_digital_divide_project\data\raw"
processed_folder = r"D:\Source\repo\flemms_digital_divide_project\data\processed"

hh_processed_folder = os.path.join(processed_folder, "households")
ind_processed_folder = os.path.join(processed_folder, "individuals")
os.makedirs(hh_processed_folder, exist_ok=True)
os.makedirs(ind_processed_folder, exist_ok=True)

# 2. Find files
file_pattern = os.path.join(raw_folder, "**", "*.csv")
data_files = glob.glob(file_pattern, recursive=True)

# 3. Process and save immediately
print("Starting partitioning process...")
for file in data_files:
    parent_folder = os.path.basename(os.path.dirname(file))
    filename = os.path.basename(file).upper()
    
    if "(1)" in parent_folder:
        continue
        
    print(f"Processing: {parent_folder} -> {filename}...")
    
    try:
        df = pd.read_csv(file, on_bad_lines='skip', encoding='latin-1', low_memory=False)
        df['source_folder'] = parent_folder
        df['source_file'] = filename
        
        safe_filename = f"{parent_folder}_{filename}.csv"
        
        if any(keyword in filename for keyword in ["HOUSEHOLD", "RT01", "RTF1", "HOUSING"]):
            out_path = os.path.join(hh_processed_folder, safe_filename)
            df.to_csv(out_path, index=False)
            print("  -> Saved to Households.")
            
        elif any(keyword in filename for keyword in ["POPULATION", "MEMBER", "RT02", "RTF2", "DEMOGRAPHIC", "INDIVIDUAL", "2003"]):
            out_path = os.path.join(ind_processed_folder, safe_filename)
            df.to_csv(out_path, index=False)
            print("  -> Saved to Individuals.")
            
        del df
        gc.collect()
        
    except Exception as e:
        print(f"Error loading {file}: {e}")

print("Partitioning Complete! The folders are now populated.")

Starting partitioning process...
Processing: PHL-NSO-FLEMMS-1994-PUF -> FLEMMS PUF 1994 HOUSEHOLD.CSV...
  -> Saved to Households.
Processing: PHL-NSO-FLEMMS-1994-PUF -> FLEMMS PUF 1994 POPULATION.CSV...
  -> Saved to Individuals.
Processing: PHL-PSA-FLEMMS-2003-PUF -> FLEMMS PUF 2003.CSV...
  -> Saved to Individuals.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS OSY PUF 2008 RT01 - ALL MEMBERS.CSV...
  -> Saved to Households.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS OSY PUF 2008 RT02 - 10-64 YEAR OLD.CSV...
  -> Saved to Individuals.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS PUF 2008 RT01 - ALL MEMBERS.CSV...
  -> Saved to Households.
Processing: PHL-PSA-FLEMMS-2008-PUF -> FLEMMS PUF 2008 RT02 - 10-64 YEARS OLD.CSV...
  -> Saved to Individuals.
Processing: Standard -> FLEMMS PUF 2013 STANDARD RT01 - HOUSEHOLD MEMBERS.CSV...
  -> Saved to Households.
Processing: Standard -> FLEMMS PUF 2013 STANDARD RT02 - INDIVIDUAL.CSV...
  -> Saved to Individuals.
Processing: Urban Rural -> 

In [4]:
import pandas as pd
import os

ind_processed_folder = r"D:\Source\repo\flemms_digital_divide_project\data\processed\individuals"

# Modified to be case-insensitive for .csv / .CSV
available_files = [f for f in os.listdir(ind_processed_folder) if f.lower().endswith('.csv')]

if not available_files:
    print("No CSV files found in the individuals folder.")
else:
    sample_file = available_files[0]
    file_path = os.path.join(ind_processed_folder, sample_file)

    print(f"--- Prototyping ETL on: {sample_file} ---\n")
    
    df = pd.read_csv(file_path, low_memory=False)
    initial_rows = len(df)
    initial_cols = len(df.columns)

    # Clean headers
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

    # Deduplicate
    df = df.drop_duplicates()
    dropped_duplicates = initial_rows - len(df)

    # Drop fully empty columns
    df = df.dropna(axis=1, how='all')
    dropped_cols = initial_cols - len(df.columns)

    print("=== POST-CLEANING OVERVIEW ===")
    print(f"Duplicate Rows Removed: {dropped_duplicates}")
    print(f"Completely Empty Columns Dropped: {dropped_cols}")
    print(f"Current Rows: {df.shape[0]}")
    print(f"Current Columns: {df.shape[1]}\n")
    
    print("Cleaned Column Names:")
    print(df.columns.tolist())

    print("\n--- First 5 Rows of Cleaned Data ---")
    display(df.head())

--- Prototyping ETL on: PHL-NSO-FLEMMS-1994-PUF_FLEMMS PUF 1994 POPULATION.CSV ---

=== POST-CLEANING OVERVIEW ===
Duplicate Rows Removed: 0
Completely Empty Columns Dropped: 0
Current Rows: 134563
Current Columns: 86

Cleaned Column Names:
['ï»¿fn', 'regn', 'prov', 'mun', 'bgy', 'urb', 'bstr', 'hcn', 'prrcd', 'lno', 'rel', 'sex', 'age', 'mstat', 'ocwind', 'ablerw', 'heduc', 'hsloc', 'colloc', 'editinst', 'cursch', 'leduc', 'modetr', 'rnasch', 'backsch', 'activ', 'memorg', 'kindorg', 'vocind', 'voccode', 'vocnst', 'locinst', 'hsreq', 'apply', 'aoccup', 'wattend', 'rwattend', 'wvoc', 'busoc', 'poccup', 'kbus', 'clawor', 'rnemp', 'resp', 'rnewsy', 'rnewpw', 'knews', 'trnews', 'rmagpw', 'rcompw', 'rbookpw', 'lrady', 'lradpw', 'kradiop', 'tlradio', 'wtvy', 'wtvpw', 'ktvp', 'twtv', 'wvcrpw', 'kvideo', 'lvideo', 'wmoviepw', 'kmovie', 'lmovie', 'medce', 'name', 'address', 'birth', 'hduc2', 'amtr', 'amts', 'explain', 'rescode', 'rf1_unadj', 'rfajp', 'agerc', 'hgcrc', 'osyind', 'lcarc', 'vocrc'

,ï»¿fn,regn,prov,mun,bgy,urb,bstr,hcn,prrcd,lno,...,agerc,hgcrc,osyind,lcarc,vocrc,mjoapply,poccrc,llrc,source_folder,source_file
0,458,0,39,1,1291,1,1,1,39,1,...,08,5,2,,05,,7,4,PHL-NSO-FLEMMS-1994-PUF,FLEMMS PUF 1994 POPULATION.CSV
1,458,0,39,1,1291,1,1,1,39,2,...,06,3,1,,,,,0,PHL-NSO-FLEMMS-1994-PUF,FLEMMS PUF 1994 POPULATION.CSV
2,458,0,39,1,1291,1,1,1,39,3,...,00,,2,,,,,9,PHL-NSO-FLEMMS-1994-PUF,FLEMMS PUF 1994 POPULATION.CSV
3,458,0,39,1,1291,1,1,1,39,4,...,00,,2,,,,,9,PHL-NSO-FLEMMS-1994-PUF,FLEMMS PUF 1994 POPULATION.CSV
4,458,0,39,1,1291,1,1,2,39,1,...,09,2,2,,,,5,0,PHL-NSO-FLEMMS-1994-PUF,FLEMMS PUF 1994 POPULATION.CSV
